In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from keras.models import Sequential
from keras.layers import Dense, LSTM
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
from scipy.stats import ttest_1samp
from sklearn import metrics
from sklearn.metrics import r2_score, mean_squared_error



In [2]:
df=pd.read_csv("bitcoin.csv")
df.head()

,Date,Open,High,Low,Close,Volume,Month,Day,Year,Daily_Return,7-day MA,30-day MA,30-day Volatility
0,2015-01-01,320.434998,320.434998,314.002991,314.248993,8036550,1,1,2015,NaN,NaN,NaN,NaN
1,2015-01-02,314.079010,315.838989,313.565002,315.032013,7860650,1,2,2015,0.002492,NaN,NaN,NaN
2,2015-01-03,314.846008,315.149994,281.082001,281.082001,33054400,1,3,2015,-0.107767,NaN,NaN,NaN
3,2015-01-04,281.145996,287.230011,257.612000,264.195007,55629100,1,4,2015,-0.060079,NaN,NaN,NaN
4,2015-01-05,265.084015,278.341003,265.084015,274.473999,43962800,1,5,2015,0.038907,NaN,NaN,NaN


In [3]:
X=df.drop(['Close','Date'],axis=1)
y=df['Close']

In [4]:
# splitting X and y into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

In [5]:
# import the regressor
from sklearn.ensemble import RandomForestRegressor
  
 # create regressor object
model = RandomForestRegressor(n_estimators = 100, random_state = 0)
  
# fit the regressor with x and y data
model.fit(X_train, y_train) 
y_pred = model.predict(X_test)

In [12]:
print('Mean Absolute Error:', metrics.mean_absolute_error(y_test, y_pred))
print('Mean Squared Error:', metrics.mean_squared_error(y_test, y_pred))
print('Root Mean Squared Error:', np.sqrt(metrics.mean_squared_error(y_test, y_pred)))
print('r2_score:',r2_score(y_test,y_pred))

Mean Absolute Error: 197.1821658662946
Mean Squared Error: 192346.5373522021
Root Mean Squared Error: 438.573297582288
r2_score: 0.9995391207161998


In [13]:
import scipy.stats as stats

# Step 6: Hypothesis Test
# Null Hypothesis (H0): The mean difference between predicted and actual values is 0 (no difference).
# Alternative Hypothesis (H1): The mean difference between predicted and actual values is not 0 (significant difference).

# Calculate the difference between predicted and actual values
difference = y_test - y_pred.flatten()

# Perform a paired t-test
t_stat, p_value = stats.ttest_1samp(difference, 0)

# Display the t-statistic and p-value
print(f"T-statistic: {t_stat}")
print(f"P-value: {p_value}")

# Step 7: Conclusion of Hypothesis Test
if p_value < 0.05:
    print("Reject the null hypothesis: There is a significant difference between predicted and actual values.")
else:
    print("Fail to reject the null hypothesis: No significant difference between predicted and actual values.")

T-statistic: -0.8516599740521339
P-value: 0.3946321953398577
Fail to reject the null hypothesis: No significant difference between predicted and actual values.


## LSTM

In [24]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Step 1: Load Data
df = pd.read_csv('bitcoin.csv')  # Replace 'bitcoin.csv' with actual file name
df.dropna(inplace=True)

# Step 2: Prepare Features and Target
X = df.drop(['Close', 'Date'], axis=1)
y = df['Close']

# Scale the features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Define lookback period
lookback = 10

# Create sequences
def create_sequences(data, target, lookback):
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i - lookback:i])  # Create sequence
        y.append(target[i])  # Corresponding target
    return np.array(X), np.array(y)

X_seq, y_seq = create_sequences(X_scaled, y.values, lookback)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_seq, y_seq, test_size=0.25, random_state=0)

# Step 3: Build the LSTM Model
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    LSTM(50, return_sequences=False),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

# Step 4: Train the Model
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=20, batch_size=32)

# Step 5: Evaluate the Model
y_pred = model.predict(X_test)

# Flatten predictions for evaluation
y_pred = y_pred.flatten()

# Calculate metrics
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(y_test - y_pred))

print("R^2 Score:", r2)
print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("Mean Absolute Error (MAE):", mae)


/home/asif/Projects/Bitcoin/myenv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_8 (LSTM)                   │ (None, 10, 50)         │        12,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_9 (LSTM)                   │ (None, 50)             │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,651 (127.54 KB)

 Trainable params: 32,651 (127.54 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - loss: 707783488.0000 - val_loss: 780585088.0000
Epoch 2/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - loss: 720998080.0000 - val_loss: 780404928.0000
Epoch 3/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 718200384.0000 - val_loss: 780238336.0000
Epoch 4/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 738147456.0000 - val_loss: 780076416.0000
Epoch 5/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 775041984.0000 - val_loss: 779915840.0000
Epoch 6/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 773245632.0000 - val_loss: 779748864.0000
Epoch 7/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 691534208.0000 - val_loss: 779586112.0000
Epoch 8/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 687072448.0000 - val_loss: 779420864.0000
Epoch 9/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 696805440.0000 - val_loss: 779258560.0000
Epoch 10/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 713309248.0000 - val_loss: 7790

T-statistic: 27.39487458950817
P-value: 6.0990308887530206e-120
Reject the null hypothesis: There is a significant difference between predicted and actual values.


In [17]:
# Step 2: Prepare Features and Target
df.dropna(inplace=True)
features = ['Daily_Return', '7-day MA', '30-day MA']
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[features + ['Close']])

# Create sequences
def create_sequences(data, lookback):
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i-lookback:i, :-1])
        y.append(data[i, -1])
    return np.array(X), np.array(y)

lookback = 10
X, y = create_sequences(scaled_data, lookback)

# Train-test split
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Step 3: Build the LSTM Model
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    LSTM(50, return_sequences=False),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

# Step 4: Train the Model
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=20, batch_size=32)



/home/asif/Projects/Bitcoin/myenv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 10, 50)         │        10,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 50)             │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,051 (121.29 KB)

 Trainable params: 31,051 (121.29 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - loss: 0.0184 - val_loss: 0.0020
Epoch 2/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0011 - val_loss: 0.0026
Epoch 3/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0011 - val_loss: 0.0021
Epoch 4/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 7.5882e-04 - val_loss: 0.0016
Epoch 5/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 9.1045e-04 - val_loss: 0.0015
Epoch 6/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 6.5063e-04 - val_loss: 0.0021
Epoch 7/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0010 - val_loss: 0.0039
Epoch 8/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 7.9388e-04 - val_loss: 0.0012
Epoch 9/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 8.2389e-04 - val_loss: 0.0013
Epoch 10/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 4.9748e-04 - val_loss: 0.0011
Epoch 11/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 4.4753e-04 - val_loss: 0.0020
Epoch 12/20
88/88 ━━━━━━━━━━━━━━━━━

In [18]:
print('Mean Absolute Error:', metrics.mean_absolute_error(y_test, y_pred))
print('Mean Squared Error:', metrics.mean_squared_error(y_test, y_pred))
print('Root Mean Squared Error:', np.sqrt(metrics.mean_squared_error(y_test, y_pred)))
print('r2_score:',r2_score(y_test,y_pred))

ValueError: Found input variables with inconsistent numbers of samples: [704, 710]

In [19]:
print("Shape of y_test:", y_test.shape)
print("Shape of y_pred:", y_pred.shape)


Shape of y_test: (704,)
Shape of y_pred: (710, 1)


In [27]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Step 1: Load Data
# Replace this with actual data file
df = pd.read_csv('bitcoin.csv')  # Replace 'your_data.csv' with actual file name

# Ensure "Daily_Return," "7-day MA," and "30-day MA" are calculated
df['Daily_Return'] = df['Close'].pct_change()
df['7-day MA'] = df['Close'].rolling(window=7).mean()
df['30-day MA'] = df['Close'].rolling(window=30).mean()
df.dropna(inplace=True)

# Step 2: Prepare Features and Target
features = ['Daily_Return', '7-day MA', '30-day MA']
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[features + ['Close']])

# Create sequences
def create_sequences(data, lookback):
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i-lookback:i, :-1])
        y.append(data[i, -1])
    return np.array(X), np.array(y)

lookback = 10
X, y = create_sequences(scaled_data, lookback)

# Train-test split
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Step 3: Build the LSTM Model
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    LSTM(50, return_sequences=False),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

# Step 4: Train the Model
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=20, batch_size=32)

# Step 5: Evaluate the Model
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print("R^2 Score:", r2)
print("Mean Squared Error:", mse)

/home/asif/Projects/Bitcoin/myenv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_12 (LSTM)                  │ (None, 10, 50)         │        10,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_13 (LSTM)                  │ (None, 50)             │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,051 (121.29 KB)

 Trainable params: 31,051 (121.29 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - loss: 0.0105 - val_loss: 0.0032
Epoch 2/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0012 - val_loss: 0.0024
Epoch 3/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0012 - val_loss: 0.0014
Epoch 4/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 8.8281e-04 - val_loss: 0.0015
Epoch 5/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0011 - val_loss: 0.0017
Epoch 6/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 8.2420e-04 - val_loss: 0.0016
Epoch 7/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 5.7179e-04 - val_loss: 0.0015
Epoch 8/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 6.4589e-04 - val_loss: 0.0015
Epoch 9/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 5.8593e-04 - val_loss: 0.0015
Epoch 10/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 5.6131e-04 - val_loss: 0.0033
Epoch 11/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 5.6678e-04 - val_loss: 9.1224e-04
Epoch 12/20
88/88 ━━━━━━━━━━━━━

In [28]:
import scipy.stats as stats

# Step 6: Hypothesis Test
# Null Hypothesis (H0): The mean difference between predicted and actual values is 0 (no difference).
# Alternative Hypothesis (H1): The mean difference between predicted and actual values is not 0 (significant difference).

# Calculate the difference between predicted and actual values
difference = y_test - y_pred.flatten()

# Perform a paired t-test
t_stat, p_value = stats.ttest_1samp(difference, 0)

# Display the t-statistic and p-value
print(f"T-statistic: {t_stat}")
print(f"P-value: {p_value}")

# Step 7: Conclusion of Hypothesis Test
if p_value < 0.05:
    print("Reject the null hypothesis: There is a significant difference between predicted and actual values.")
else:
    print("Fail to reject the null hypothesis: No significant difference between predicted and actual values.")

T-statistic: -13.55428415351916
P-value: 2.3761642291017e-37
Reject the null hypothesis: There is a significant difference between predicted and actual values.
